# 01 — Validation du Pipeline RAG

## Objectif

Ce notebook vérifie que le pipeline RAG fonctionne correctement :

1. Chargement du vector store (index FAISS)
2. Recherche des documents pertinents (retrieval)
3. Génération d'une réponse par le LLM
4. Évaluation avec DeepEval

**Pourquoi cette étape est importante**

Avant de lancer des expériences comparatives, il faut s'assurer que
tous les composants communiquent correctement. Ce notebook sert de
test d'intégration : si tout passe, le projet est prêt pour la phase
d'expérimentation.

## 1. Import des modules

On importe directement les modules depuis `src/` :
- `retriever` : chargement de l'index et recherche
- `llm_chain` : construction du prompt et génération
- `config` : chargement de la configuration
- `evaluation_judge` : création du juge DeepEval

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "src").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))

from config import load_config, RetrievalConfig, LLMConfig, EmbeddingConfig
from retriever import retrieve_documents
from llm_chain import generate_answer, format_sources
from evaluation_judge import create_judge

import pandas as pd
import time

print("✅ Imports réussis")

✅ Imports réussis


## 2. Configuration

On charge la configuration depuis `config.yaml`.
C'est ici qu'on peut modifier le LLM, l'embedding ou le retrieval
avant de lancer les tests.

In [2]:
cfg = load_config()

print(f"🔤 Embedding : {cfg.embedding.provider}:{cfg.embedding.model}")
print(f"🤖 LLM       : {cfg.llm.provider}:{cfg.llm.model}")
print(f"📄 Retrieval : top_k={cfg.retrieval.top_k}")

🔤 Embedding : huggingface:BAAI/bge-m3
🤖 LLM       : openrouter:google/gemini-2.5-flash
📄 Retrieval : top_k=6


## 3. Test du retrieval

On pose une question simple et on observe les documents récupérés.

**Ce que fait le code :**
- Il transforme la question en vecteur via le modèle d'embedding
- Il compare ce vecteur à tous les vecteurs de l'index FAISS
- Il retourne les top_k chunks les plus similaires

**Comment interpréter :**
- Le score (similarité) doit être > 0.5 pour un bon match
- Les chunks doivent parler du même sujet que la question
- Si les chunks sont hors-sujet, l'embedding ou l'index est en cause

In [3]:
question = "Peut-on demander une prolongation du mémoire ?"
retrieval_cfg = RetrievalConfig(top_k=3, max_distance=1.5)

documents, scores = retrieve_documents(question, retrieval_cfg)

print(f"📝 Question : {question}")
print(f"📄 {len(documents)} chunk(s) récupéré(s)\n")

for i, (doc, score) in enumerate(zip(documents, scores)):
    print(f"--- Chunk {i+1} (similarité: {score:.4f}) ---")
    print(doc.page_content[:200])
    print()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

📝 Question : Peut-on demander une prolongation du mémoire ?
📄 3 chunk(s) récupéré(s)

--- Chunk 1 (similarité: 0.5914) ---
La  Partie  B  doit  déposer  un  mémoire  écrit  dans  la  langue  prescrite  par  le  programme  
de
 
formation
 
sous
 
30
 
jours
 
après
 
la
 
ﬁn
 
du
 
stage.
 
Si
 
la
 
Partie
 
B
 
souhaite

--- Chunk 2 (similarité: 0.5892) ---
3.2.4.  Mémoire  La  Partie  B  doit  déposer  un  mémoire  écrit  dans  la  langue  prescrite  par  le  programme  de  formation  sous  30  jours  après  la  fin  du  stage.  Si  la  Partie  B  souha

--- Chunk 3 (similarité: 0.5505) ---
prolongation  sont  de  1.000.000  (un  million)  VND  par  mois  de  prolongation  des  
études
 
(Sauf
 
dans
 
le
 
cas
 
où
 
la
 
formation
 
est
 
prolongée
 
par
 
l’IFI).
 
Ces
 
frais
 
ne
 




## 4. Test de la génération

Le LLM reçoit le contexte (chunks récupérés) + la question, et génère une réponse.

**Pourquoi c'est important :**
- C'est le cœur du RAG : la qualité de la réponse dépend du contexte fourni
- On vérifie que le LLM répond en français et utilise bien les documents

In [4]:
start = time.time()
answer = generate_answer(question, documents, cfg.llm)
elapsed = time.time() - start

print(f"🤖 Réponse générée en {elapsed:.1f}s :\n")
print(answer)
print()

sources = format_sources(documents)
if sources:
    print(f"📚 Sources : {', '.join(sources)}")

🤖 Réponse générée en 3.4s :

Oui, vous pouvez demander une prolongation du délai de dépôt du mémoire. Vous devez soumettre une demande écrite au Service de scolarité et des Affaires étudiantes avant la date limite, avec des documents justificatifs. La Partie A examinera et décidera de votre demande.

📚 Sources : Page 2, Page 3, Page 4


## 5. Test DeepEval

DeepEval mesure automatiquement la qualité de la réponse générée.

**Métriques utilisées :**
- **Faithfulness** (fidélité) : la réponse est-elle fidèle au contexte ?
- **Answer Relevancy** (pertinence) : la réponse répond-elle à la question ?

**Comment interpréter :**
- Score > 0.75 : bonne performance
- Score < 0.5 : problème à investiguer
- Un score parfait (1.0) est rare et pas nécessairement attendu

In [5]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric

# Créer un cas de test
test_case = LLMTestCase(
    input=question,
    actual_output=answer,
    retrieval_context=[doc.page_content for doc in documents],
)

# Initialiser le juge
judge = create_judge(provider=cfg.evaluation.provider, model=cfg.evaluation.model)
print(f"⚖️  Juge : {judge.get_model_name()}\n")

⚖️  Juge : qwen2.5:3b



In [6]:
# Mesure Faithfulness
faithfulness = FaithfulnessMetric(threshold=0.75, model=judge, include_reason=True)
faithfulness.measure(test_case)

print(f"📊 Faithfulness : {faithfulness.score:.4f}")
print(f"   {'✅ Passé' if faithfulness.score >= 0.75 else '❌ Échoué'}")
print(f"   Raison : {faithfulness.reason}")

Output()

ValueError: Evaluation LLM outputted an invalid JSON. Please use a better evaluation model.

In [7]:
# Mesure AnswerRelevancy
relevancy = AnswerRelevancyMetric(threshold=0.75, model=judge, include_reason=True)
relevancy.measure(test_case)

print(f"📊 AnswerRelevancy : {relevancy.score:.4f}")
print(f"   {'✅ Passé' if relevancy.score >= 0.75 else '❌ Échoué'}")
print(f"   Raison : {relevancy.reason}")

Output()

📊 AnswerRelevancy : 0.7500
   ✅ Passé
   Raison : The score is 0.75 because while the answer addresses the topic of requesting an extension for a thesis, it does so by providing information about academic procedures rather than directly addressing features or requirements related to submitting a thesis in the original input.


## 6. Test sur plusieurs questions

On répète le processus sur 3 questions pour vérifier la stabilité.
C'est un test de non-régression avant les expériences comparatives.

In [8]:
questions = [
    "Quels sont les critères d'admission en Master SIM ?",
    "Quel est le montant des frais de prolongation ?",
    "Comment se déroule la soutenance de mémoire ?",
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"📝 {q}")
    print(f"{'='*60}")

    docs, scores = retrieve_documents(q, retrieval_cfg)
    ans = generate_answer(q, docs, cfg.llm)
    print(f"\n🤖 {ans[:200]}...")
    print(f"📄 {len(docs)} chunks récupérés")


📝 Quels sont les critères d'admission en Master SIM ?


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


🤖 Information non trouvée dans les règlements officiels....
📄 3 chunks récupérés

📝 Quel est le montant des frais de prolongation ?


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


🤖 Les frais de prolongation sont de 1.000.000 VND par mois de prolongation des études....
📄 3 chunks récupérés

📝 Comment se déroule la soutenance de mémoire ?


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


🤖 Information non trouvée dans les règlements officiels....
📄 3 chunks récupérés


## 7. Conclusion

Le pipeline RAG est validé si :

1. Le retrieval retourne des chunks pertinents (score > 0.5)
2. Le LLM génère une réponse en français, cohérente avec le contexte
3. DeepEval s'exécute sans erreur et produit des métriques

Le projet est prêt pour les expériences comparatives.

---
**Prochaine étape :** 02_compare_llms.ipynb